# Convergence diagnostics in scqubits

In scqubits, each qubit's Hamiltonian acts on an (in general) infinite-dimensional
Hilbert space and is represented numerically as a *finite matrix* by truncating
the basis it is expressed in: the transmon in a charge basis (`ncut`), fluxonium
in a harmonic-oscillator basis (`cutoff`), the grid-based qubits by discretizing a
flux coordinate on a finite box. The computed quantities -- energies, matrix
elements, coherence times -- are only as accurate as that truncated representation
allows.

`estimate_convergence` answers, for the cutoff you chose: **how accurate is the
result, and what should you do if it is not accurate enough?** The guiding idea
is *falsification*: a convergence test can only ever **dismiss** convergence. It
refines the cutoff and looks for the spectrum still moving, a state spilling over
the basis boundary, or a refinement series that refuses to contract; when it
finds one, it has caught a clearly-wrong result. When it finds none, all it can
honestly say is "not dismissed" -- never an outright guarantee. The most
actionable signal is therefore a `distrust` verdict.

The returned `ConvergenceReport` prints itself: `print(report)` (its `summary()`
method) gives a compact, human-readable rundown, used throughout this notebook.

In [1]:
import numpy as np
import scqubits as scq

## 1. A first check (moderate mode)

`moderate` mode (the default) refines the cutoff once and compares. You supply
the number of levels and an absolute target `target_abs_GHz` against which each
level's estimated error is graded. Printing the report shows the aggregate
verdict, the per-level detail, the per-channel breakdown, and recommendations.

In [2]:
tmon = scq.Transmon(EJ=20.0, EC=0.3, ng=0.0, ncut=31, truncated_dim=6)

report = tmon.estimate_convergence(n_levels=5, target_abs_GHz=1e-4)
print(report)

aggregate: maybe_converged   (worst: level 0)

  lvl   status            channel       err (GHz)   via
    0   maybe_converged   charge_tail    7.11e-15   one_step
    1   maybe_converged   charge_tail    1.85e-13   one_step
    2   maybe_converged   charge_tail    2.47e-13   one_step
    3   maybe_converged   charge_tail    4.97e-14   one_step
    4   maybe_converged   charge_tail    4.44e-14   one_step

  error by channel (GHz): charge_tail=2.47e-13


## 2. Reading the per-level verdict

The **status** is the verdict -- one of five values, ordered from best (left) to worst (right):

> `likely_converged`  >  `maybe_converged`  >  `marginal`  >  `unverified`  >  `distrust`

`likely_converged` and `maybe_converged` both mean "not dismissed" (by the strict ratio test and the moderate one-step refinement, respectively); `marginal` is borderline against the target; `unverified` is a cheap pass or an unassessed level; and `distrust` is the one verdict that states a fact -- a test actively caught a wrong result.

Each `LevelVerdict` also carries:

- an **abs_err_est_GHz** (and an `eps_gap_est` in observed-gap scope);
- a **truncation_channel** -- the physical error source (`charge_tail`, `HO_tail`, `FD_box`, `FD_stencil`, `composite_coupling`);
- an **estimator_method** and any **warnings**;
- a **checks** tuple recording which falsification tests ran for the level (each `pass`, `fail`, or `not_applicable`).

The verdict name itself carries the confidence; to see *why* a level earned it, read its **checks** record (below). There is no separate evidence field. The fields are available programmatically, too:

In [3]:
v = report.level(0)
print("level_index        :", v.level_index)
print("status             :", v.status)
print("abs_err_est_GHz    :", v.abs_err_est_GHz)
print("truncation_channel :", v.truncation_channel)
print("estimator_method   :", v.estimator_method)
print("status_scope       :", v.status_scope)
print("checks             :", [(chk.name, chk.status) for chk in v.checks])

level_index        : 0
status             : maybe_converged
abs_err_est_GHz    : 7.105427357601002e-15
truncation_channel : charge_tail
estimator_method   : one_step
status_scope       : absolute
checks             : [('asymptoticity', 'not_applicable'), ('boundary', 'pass'), ('monotonicity', 'pass')]


## 3. Cheap mode -- a cheap perturbative estimate (no refinement)

`cheap` mode skips refinement and instead uses a cheap, basis-specific
perturbative estimator (for the transmon: the charge finite-tail Green-function
estimate). It can still *dismiss* a level -- a kept state reaching the basis
boundary is caught here -- but a level it fails to dismiss earns at best
`unverified`: cheap mode makes no verification claim.

In [4]:
print(tmon.estimate_convergence(n_levels=5, mode="cheap", target_abs_GHz=1e-4))

aggregate: unverified   (worst: level 0)

  lvl   status       channel       err (GHz)   via
    0   unverified   charge_tail    1.63e-86   finite_tail_resolvent
    1   unverified   charge_tail    9.48e-85   finite_tail_resolvent
    2   unverified   charge_tail    2.57e-83   finite_tail_resolvent
    3   unverified   charge_tail    4.29e-82   finite_tail_resolvent
    4   unverified   charge_tail    4.80e-81   finite_tail_resolvent


## 4. Strict mode -- a two-step ratio test

`strict` mode refines twice and runs an asymptoticity (ratio) test. A level that
passes earns `likely_converged` -- the strongest verdict available; one whose
refinement movements are not shrinking is dismissed to `distrust` rather than
softened to `marginal`.

In [5]:
print(tmon.estimate_convergence(n_levels=4, mode="strict", target_abs_GHz=1e-4))

aggregate: likely_converged   (worst: level 0)

  lvl   status             channel       err (GHz)   via
    0   likely_converged   charge_tail    9.95e-14   ratio_test_noise_floor
    1   likely_converged   charge_tail    3.80e-13   ratio_test_noise_floor
    2   likely_converged   charge_tail    3.38e-13   ratio_test_noise_floor
    3   likely_converged   charge_tail    3.32e-13   ratio_test_noise_floor

  error by channel (GHz): charge_tail=2.45e-13


## 5. Catching an under-resolved cutoff

At a deliberately small `ncut` the higher levels reach the charge boundary. The
verdict is `distrust`, the affected levels carry a `boundary_probability_large`
warning (their dropped tail is non-perturbative), and the recommendations are
channel-specific.

In [6]:
small = scq.Transmon(EJ=20.0, EC=0.3, ng=0.0, ncut=5, truncated_dim=6)
print(small.estimate_convergence(n_levels=5, target_abs_GHz=1e-6))

aggregate: distrust   (worst: level 0)

  lvl   status     channel       err (GHz)   via
    0   distrust   charge_tail    3.81e-04   one_step
      checks 0: asymptoticity=n/a(strict mode only)  boundary=pass(P_edge=0.00011)  monotonicity=pass
    1   distrust   charge_tail    5.62e-03   one_step   [boundary_probability_large]
      checks 1: asymptoticity=n/a(strict mode only)  boundary=fail(P_edge=0.0014)  monotonicity=pass
    2   distrust   charge_tail    3.83e-02   one_step   [boundary_probability_large]
      checks 2: asymptoticity=n/a(strict mode only)  boundary=fail(P_edge=0.008)  monotonicity=pass
    3   distrust   charge_tail    1.60e-01   one_step   [boundary_probability_large]
      checks 3: asymptoticity=n/a(strict mode only)  boundary=fail(P_edge=0.028)  monotonicity=pass
    4   distrust   charge_tail    4.55e-01   one_step   [boundary_probability_large]
      checks 4: asymptoticity=n/a(strict mode only)  boundary=fail(P_edge=0.066)  monotonicity=pass

  error by ch

## 6. Transition-frequency error estimates

For a transition `k -> j` the triangle inequality bounds the transition error by
the sum of the two level error estimates. Each verdict carries these for the
other requested levels.

In [7]:
v0 = report.level(0)
for (i, j), te in sorted(v0.transition_err_est_GHz.items()):
    print(f"  transition {i}->{j}: <= {te:.2e} GHz")

  transition 0->1: <= 1.92e-13 GHz
  transition 0->2: <= 2.54e-13 GHz
  transition 0->3: <= 5.68e-14 GHz
  transition 0->4: <= 5.15e-14 GHz


## 7. Derived quantities -- wavefunctions, matrix elements, coherence

Eigenvalues can converge faster than eigenvectors or derived quantities. Request
derived channels to check them too; each is returned as a sub-report under
`report.derived` and is shown indented when you print the parent report.
Coherence is assessed rate-first (a channel whose rate sits at the noise floor is
flagged rather than turned into a lifetime).

In [8]:
report_d = tmon.estimate_convergence(
    n_levels=4,
    target_abs_GHz=1e-4,
    include_derived=True,
    derived_quantities=["wavefunctions", "matrix_elements", "coherence"],
)
print(report_d)

aggregate: maybe_converged   (worst: level 0)

  lvl   status            channel       err (GHz)   via
    0   maybe_converged   charge_tail    2.13e-14   one_step
    1   maybe_converged   charge_tail    2.45e-13   one_step
    2   maybe_converged   charge_tail    1.84e-13   one_step
    3   maybe_converged   charge_tail    5.33e-14   one_step

  error by channel (GHz): charge_tail=2.45e-13
  derived [wavefunctions]:
    aggregate: maybe_converged   (worst: level 0)

      lvl   status            channel        rel_chg   via
        0   maybe_converged   charge_tail   4.44e-16   wavefunction_overlap
        1   maybe_converged   charge_tail   0.00e+00   wavefunction_overlap
        2   maybe_converged   charge_tail   4.44e-16   wavefunction_overlap
        3   maybe_converged   charge_tail   0.00e+00   wavefunction_overlap
  derived [matrix_elements]:
    aggregate: maybe_converged   (worst: level 0)

      lvl   status            channel        rel_chg   via
        0   maybe_converg

See documentation for details.
This warning can be disabled by executing:
scqubits.settings.T1_DEFAULT_WARNING=False

 c:\users\drjen\coding\scqubits\scqubits\core\noise.py: 1248

## 8. Observed-gap scope

Instead of an absolute GHz target, you can grade each level's error relative to
its local isolation gap (a dimensionless `eps_gap_est`). A buffer level is
diagonalized automatically so the topmost requested level still has an upper gap.

In [9]:
print(tmon.estimate_convergence(n_levels=4, scope="observed_gap_scale", target_gap_rel=1e-3))

aggregate: maybe_converged   (worst: level 0)

  lvl   status            channel       err (GHz)    gap_rel   via
    0   maybe_converged   charge_tail    7.11e-15   1.07e-15   one_step
    1   maybe_converged   charge_tail    1.85e-13   2.94e-14   one_step
    2   maybe_converged   charge_tail    2.47e-13   4.18e-14   one_step
    3   maybe_converged   charge_tail    4.97e-14   9.08e-15   one_step

  error by channel (GHz): charge_tail=2.47e-13


## 9. Other qubits and their truncation channels

The same `estimate_convergence` call works across qubit types; the reported
`truncation_channel` identifies the physical error source. (A multi-dimensional
charge basis such as `FluxQubit` has no closed-form cheap tail estimate, so its
`cheap` mode falls back to a boundary diagnostic -- use `moderate` or `strict`
for an empirical estimate.)

In [10]:
flx = scq.Fluxonium(EJ=8.9, EC=2.5, EL=0.5, flux=0.5, cutoff=110, truncated_dim=6)
print("Fluxonium (HO_tail), moderate mode:")
print(flx.estimate_convergence(n_levels=4, target_abs_GHz=1e-4))
print("\nFluxonium cheap mode (HO finite-window block-resolvent):")
print(flx.estimate_convergence(n_levels=4, mode="cheap", target_abs_GHz=1e-4))

Fluxonium (HO_tail), moderate mode:


aggregate: maybe_converged   (worst: level 0)

  lvl   status            channel   err (GHz)   via
    0   maybe_converged   HO_tail    8.24e-14   one_step   [cluster_index_ambiguity]
    1   maybe_converged   HO_tail    8.24e-14   one_step   [cluster_index_ambiguity]
    2   maybe_converged   HO_tail    1.16e-12   one_step
    3   maybe_converged   HO_tail    1.88e-12   one_step

  error by channel (GHz): HO_tail=1.88e-12

Fluxonium cheap mode (HO finite-window block-resolvent):


aggregate: unverified   (worst: level 0)

  lvl   status       channel   err (GHz)   via
    0   unverified   HO_tail    6.64e-14   finite_tail_resolvent
    1   unverified   HO_tail    1.11e-14   finite_tail_resolvent
    2   unverified   HO_tail    3.48e-13   finite_tail_resolvent
    3   unverified   HO_tail    8.45e-13   finite_tail_resolvent


In [11]:
fq = scq.FluxQubit(
    EJ1=35.0, EJ2=35.0, EJ3=0.6 * 35.0,
    ECJ1=1.0, ECJ2=1.0, ECJ3=1.0 / 0.6,
    ECg1=50.0, ECg2=50.0, ng1=0.0, ng2=0.0,
    flux=0.5, ncut=14, truncated_dim=6,
)
print("FluxQubit (charge_tail, 2D charge basis):")
print(fq.estimate_convergence(n_levels=4, target_abs_GHz=1e-4))

FluxQubit (charge_tail, 2D charge basis):


aggregate: maybe_converged   (worst: level 0)

  lvl   status            channel       err (GHz)   via
    0   maybe_converged   charge_tail    7.25e-13   one_step
    1   maybe_converged   charge_tail    1.31e-12   one_step
    2   maybe_converged   charge_tail    3.84e-13   one_step
    3   maybe_converged   charge_tail    9.38e-13   one_step

  error by channel (GHz): charge_tail=1.31e-12


A finite-difference coordinate has **two** independent error channels: the
finite box (`FD_box`, verified by widening the window at fixed spacing) and the
finite grid spacing (`FD_stencil`, refined at a fixed window). ZeroPi reports
both, plus the `charge_tail` of its theta basis; the per-level channel is the
dominant one, while `channel_breakdown_GHz` keeps the split. In strict mode the
`FD_stencil` channel is verified with Richardson `h^p` extrapolation.

In [12]:
zp = scq.ZeroPi(
    grid=scq.Grid1d(-6 * np.pi, 6 * np.pi, 100),
    EJ=10.0, EL=0.04, ECJ=20.0, EC=0.04, ng=0.1, flux=0.23, ncut=10, truncated_dim=6,
)
print("ZeroPi, moderate mode (FD_box + FD_stencil + charge_tail):")
print(zp.estimate_convergence(n_levels=3, target_abs_GHz=1e-2))
print("\nZeroPi, strict mode (Richardson for the FD stencil channel):")
print(zp.estimate_convergence(n_levels=3, mode="strict", target_abs_GHz=1e-2))

ZeroPi, moderate mode (FD_box + FD_stencil + charge_tail):


aggregate: maybe_converged   (worst: level 0)

  lvl   status            channel      err (GHz)   via
    0   maybe_converged   FD_stencil    8.98e-05   one_step   [cluster_index_ambiguity]
    1   maybe_converged   FD_stencil    8.98e-05   one_step   [cluster_index_ambiguity]
    2   maybe_converged   FD_stencil    1.36e-04   one_step

  error by channel (GHz): FD_box=3.45e-05  FD_stencil=6.02e-05  charge_tail=5.02e-05

ZeroPi, strict mode (Richardson for the FD stencil channel):


aggregate: likely_converged   (worst: level 0)

  lvl   status             channel       err (GHz)   via
    0   likely_converged   charge_tail    7.02e-05   richardson_composite   [cluster_index_ambiguity]
    1   likely_converged   charge_tail    7.02e-05   richardson_composite   [cluster_index_ambiguity]
    2   likely_converged   charge_tail    1.19e-04   richardson_composite

  error by channel (GHz): FD_box=3.45e-05  FD_stencil=4.06e-05  charge_tail=5.02e-05


## Notes and caveats

- **A convergence test only ever dismisses convergence.** A passing verdict
  (`maybe_converged`, `likely_converged`) means "not dismissed", never a
  guarantee; `distrust` is the actionable signal. Each mode caps the best verdict
  it can claim: `cheap` -> `unverified`, `moderate` -> `maybe_converged`,
  `strict` -> `likely_converged`.
- **Moderate mode is the primary, recommended check.** Cheap-mode estimates are
  cheap and useful for routing, but they make no verification claim -- moderate
  (or strict) mode is authoritative.
- **Variational monotonicity is checked automatically.** For the charge basis the
  smaller-`ncut` Hamiltonian is an exact principal submatrix of the larger, so the
  Rayleigh--Ritz min--max theorem applies: enlarging `ncut` cannot raise an ordered
  eigenvalue. A rise beyond the eigensolver noise floor therefore violates the
  variational bound, and the level is dismissed to `distrust` in any mode.
  (Harmonic-oscillator and finite-difference truncations are not exact principal
  submatrices under refinement, so they are excluded from this check.)
- **A report describes one parameter set.** Truncation convergence varies across
  a parameter sweep (e.g. fluxonium near half flux), so check the worst-case
  points: `estimate_convergence_vs_paramvals` (a single object) or
  `ParameterSweep.estimate_convergence` (a coupled sweep) do this for you.
- **Finite-difference stencil order matters for strict mode.** scqubits uses a
  7-point stencil by default (`settings.STENCIL`), so the Richardson order is
  `p = 6`. A lower-order stencil tends to realize its `h^p` regime over a wider
  usable grid range, whereas higher orders reach the round-off floor sooner; where
  the regime is not realized the asymptoticity test rejects and the estimate safely
  falls back to the one-step bound.